# Charite FoG Detection — Improved Pipeline
## Part 1: Data Loading, Preprocessing & Feature Extraction

Applies the same improved pipeline methodology as Daphnet and Figshare:
1. Applies 4th-order Butterworth bandpass filter (0.5-25 Hz for 200 Hz data)
2. Uses 4-second windows (800 samples @ 200 Hz) per literature
3. Majority voting (>50%) for window labelling
4. Computes derived signals: magnitudes and jerk from both feet
5. Feature selection via mutual information (k=80)
6. Threshold optimization via Youden's J
7. Handles dual-foot sensor data (left + right foot, acc + gyr)

Sensor placement: 3D accelerometer + 3D gyroscope on each foot (12 raw channels + 6 derived = 18 total)

In [ ]:
from __future__ import annotations

import sys, os, time, json, warnings, logging
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt
from scipy.optimize import minimize
from joblib import Parallel, delayed
from tqdm import tqdm

from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

# ── Project path ──
PROJECT_ROOT = Path(os.getcwd()).resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from loaders import ChariteDatasetLoader
from features.extractors import FeatureExtractor
from utils.pipeline_utils import (
    interpolate_missing, bandpass_filter, zscore_normalize, create_windows,
    youden_threshold, compute_metrics, get_classifiers, get_param_grids,
    prepare_fold, preprocess_features, train_and_evaluate_classifier,
    build_base_model, aggregate_results, print_results_table, print_fusion_results,
    HAS_XGB, HAS_SMOTE,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("charite")

## Constants

In [ ]:
FS = 200  # Charite sampling rate
WINDOW_SEC = 4.0
WINDOW_SAMPLES = int(WINDOW_SEC * FS)  # 800
TRAIN_OVERLAP = 0.50
TEST_OVERLAP = 0.0
LABEL_THRESH = 0.50  # majority voting
BP_LOW, BP_HIGH, BP_ORDER = 0.5, 25.0, 4
NPERSEG = min(256, WINDOW_SAMPLES)
K_FEATURES = 80
SEED = 42
N_INNER_CV = 3
N_SEARCH_ITER = 20

# Charite columns — dual-foot sensors (12 raw channels)
ACC_LEFT_COLS = ["acc_x_left_foot", "acc_y_left_foot", "acc_z_left_foot"]
GYR_LEFT_COLS = ["gyr_x_left_foot", "gyr_y_left_foot", "gyr_z_left_foot"]
ACC_RIGHT_COLS = ["acc_x_right_foot", "acc_y_right_foot", "acc_z_right_foot"]
GYR_RIGHT_COLS = ["gyr_x_right_foot", "gyr_y_right_foot", "gyr_z_right_foot"]
ALL_SENSOR_COLS = ACC_LEFT_COLS + GYR_LEFT_COLS + ACC_RIGHT_COLS + GYR_RIGHT_COLS
FOG_COL = "fog_label"

DATASET_PATH = PROJECT_ROOT / "Datasets" / "Charit\u00e9\u2013Universit\u00e4tsmedizin Berlin" / "Data Sheet 2" / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "charite_improved_results"

## Data Loading & Preprocessing

In [ ]:
def load_and_preprocess() -> Dict[int, Dict]:
    """Load Charite data, preprocess per subject/trial.
    Charite has dual-foot IMU sensors: acc(x,y,z) + gyr(x,y,z) per foot.
    Derived signals: acc_mag, gyr_mag, jerk_mag per foot (6 derived channels).
    Total: 12 raw + 6 derived = 18 channels."""
    log.info("Loading Charite dataset from %s", DATASET_PATH)
    loader = ChariteDatasetLoader(str(DATASET_PATH))
    df = loader.load_all_data(verbose=False)

    log.info("Loaded %d samples from %d subjects", len(df), df["subject_id"].nunique())

    subjects_data = {}
    for sid in sorted(df["subject_id"].unique()):
        sub_df = df[df["subject_id"] == sid]
        all_windows, all_labels = [], []

        for tid in sorted(sub_df["trial_id"].unique()):
            trial = sub_df[sub_df["trial_id"] == tid]

            if FOG_COL not in trial.columns:
                continue

            signal_raw = trial[ALL_SENSOR_COLS].values.astype(np.float64)
            labels_raw = trial[FOG_COL].values.astype(np.float64)

            # Skip trials with all NaN
            if np.all(np.isnan(signal_raw)):
                continue

            # Interpolate missing values before filtering
            signal_raw = interpolate_missing(signal_raw)
            labels_raw = np.nan_to_num(labels_raw, nan=0.0)

            # Bandpass filter
            try:
                signal_filt = bandpass_filter(signal_raw, FS, BP_LOW, BP_HIGH, BP_ORDER)
            except Exception:
                continue

            # Compute derived signals per foot
            acc_left = signal_filt[:, :3]    # indices 0,1,2
            gyr_left = signal_filt[:, 3:6]   # indices 3,4,5
            acc_right = signal_filt[:, 6:9]  # indices 6,7,8
            gyr_right = signal_filt[:, 9:12] # indices 9,10,11

            acc_mag_left = np.sqrt(np.sum(acc_left ** 2, axis=1, keepdims=True))
            gyr_mag_left = np.sqrt(np.sum(gyr_left ** 2, axis=1, keepdims=True))
            acc_mag_right = np.sqrt(np.sum(acc_right ** 2, axis=1, keepdims=True))
            gyr_mag_right = np.sqrt(np.sum(gyr_right ** 2, axis=1, keepdims=True))

            # Jerk (derivative of acceleration) per foot
            jerk_left = np.gradient(acc_left, 1.0 / FS, axis=0)
            jerk_mag_left = np.sqrt(np.sum(jerk_left ** 2, axis=1, keepdims=True))
            jerk_right = np.gradient(acc_right, 1.0 / FS, axis=0)
            jerk_mag_right = np.sqrt(np.sum(jerk_right ** 2, axis=1, keepdims=True))

            # Combine: 12 original + 6 derived = 18 channels
            # Order: [acc_l(3), gyr_l(3), acc_r(3), gyr_r(3), acc_mag_l, gyr_mag_l, jerk_mag_l, acc_mag_r, gyr_mag_r, jerk_mag_r]
            signal_full = np.hstack([signal_filt, acc_mag_left, gyr_mag_left, jerk_mag_left,
                                      acc_mag_right, gyr_mag_right, jerk_mag_right])

            # Z-score normalize per trial (ONCE only)
            signal_norm = zscore_normalize(signal_full)

            # Create windows
            w, l = create_windows(signal_norm, labels_raw, WINDOW_SAMPLES, TRAIN_OVERLAP, LABEL_THRESH)
            if len(w) > 0:
                all_windows.append(w)
                all_labels.append(l)

        if all_windows:
            subjects_data[sid] = {
                "windows": np.concatenate(all_windows, axis=0),
                "labels": np.concatenate(all_labels, axis=0),
            }
            n_fog = int(np.sum(subjects_data[sid]["labels"]))
            n_total = len(subjects_data[sid]["labels"])
            log.info("  Subject S%02d: %d windows (%d FoG, %d NoFoG)",
                     sid, n_total, n_fog, n_total - n_fog)

    return subjects_data

In [ ]:
print("=" * 90)
print("  CHARITE FoG DETECTION — IMPROVED PIPELINE")
print("=" * 90)
print(f"  Sampling rate: {FS} Hz")
print(f"  Window: {WINDOW_SEC}s ({WINDOW_SAMPLES} samples)")
print(f"  Bandpass: {BP_LOW}-{BP_HIGH} Hz")
print(f"  Label threshold: {LABEL_THRESH} (majority voting)")
print(f"  Feature selection: top {K_FEATURES} (mutual information)")
print(f"  Derived signals: acc_mag, gyr_mag, jerk_mag per foot")
print(f"  Threshold optimisation: Youden's J")
print()

t0 = time.time()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Step 1: Load & preprocess
subjects_data = load_and_preprocess()

## Feature Extraction

In [ ]:
def extract_features_for_subject(windows: np.ndarray, fs: int = FS) -> pd.DataFrame:
    """Extract features from all windows of a subject."""
    extractor = FeatureExtractor(
        sampling_rate=fs,
        extract_time=True,
        extract_frequency=True,
        extract_wavelet=True,
        extract_nonlinear=False,
    )
    rows = []
    for i in range(len(windows)):
        feats = extractor.extract_from_window(windows[i], include_magnitude=False,
                                               channel_groups=None)
        rows.append(feats)
    return pd.DataFrame(rows)


def extract_all_features(subjects_data: Dict) -> Dict[int, Dict]:
    """Extract features for all subjects in parallel."""
    log.info("Extracting features for %d subjects (parallel)...", len(subjects_data))

    def _extract(sid):
        feat_df = extract_features_for_subject(subjects_data[sid]["windows"])
        return sid, feat_df

    results = Parallel(n_jobs=-1, verbose=0)(
        delayed(_extract)(sid) for sid in tqdm(subjects_data.keys(), desc="Feature extraction")
    )

    features = {}
    for sid, feat_df in results:
        features[sid] = {"X": feat_df, "y": subjects_data[sid]["labels"]}
        log.info("  S%02d: %d windows, %d features", sid, len(feat_df), feat_df.shape[1])
    return features

In [ ]:
# Step 2: Extract features
features = extract_all_features(subjects_data)